In [ ]:
import datetime
import json
import re

import duckdb
import requests

today = datetime.datetime.now(tz=datetime.UTC).strftime("%Y-%m-%d")

conn = duckdb.connect("/Users/brianbarry/supermarket_scraping/supermarket_data.db")
conn.execute("INSTALL httpfs; LOAD httpfs;")
conn.execute("SET s3_region = 'eu-west-1';")
conn.sql("CREATE OR REPLACE SECRET secret (TYPE s3,PROVIDER credential_chain);")

In [ ]:
test = requests.get("https://www.aldi.ie/sitemap_products.xml").text
aldi_product_urls_test = re.findall(
    r"<loc>(https://www.aldi.ie/product/[^<]+)</loc>", test
)
aldi_product_ids_test = [re.findall(r"\d{18}", url)[0] for url in aldi_product_urls_test]

In [ ]:
session = requests.Session()
USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_2) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15",
]
for user_agent in USER_AGENTS:
    headers = {
        "User-Agent": f"{user_agent}",
        "Accept": "*/*",
        "Accept-Language": "en-IE",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Origin": "https://www.aldi.ie",
        "Connection": "keep-alive",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-site",
    }

    # First visit the main site to get cookies
    # session.get("https://www.aldi.ie", headers=headers)

    # Then try the API
    url = "https://api.aldi.ie/v2/products/000000000000711924"
    response = session.get(url, headers=headers)

    print(response.status_code)
    aldi_product_info = json.loads(response.text)

In [ ]:
aldi_product_info

In [ ]:
conn.close()

In [ ]:
df = conn.execute(r"""

WITH stg AS (
   SELECT *,
    TRY_CAST(
            REGEXP_EXTRACT(COALESCE(selling_size, price_comparsion_display,title), '(\d+\.?\d*)') AS DOUBLE
        ) AS unit_qty,
    LOWER(
            REGEXP_EXTRACT(COALESCE(selling_size, price_comparsion_display,title), '(\d+\.?\d*)\s*([a-zA-Z]+)', 2)
        ) AS unit_raw
    FROM stg_aldi),

   extracted AS(
   SELECT
    id,
    regexp_replace(
    regexp_replace(
        regexp_replace(
            regexp_replace(
                strip_accents(lower(title)),
            '%', ' percent', 'g'),
        '&', 'and', 'g'),
    '-', ' ', 'g'),
    '[^a-z0-9\s]', '', 'g') AS title,
    regexp_replace(
    regexp_replace(
        regexp_replace(
            regexp_replace(
                strip_accents(lower(brand)),
            '%', ' percent', 'g'),
        '&', 'and', 'g'),
    '-', ' ', 'g'),
    '[^a-z0-9\s]', '', 'g') AS brand,
    category_1,
    category_2,
    is_sale,
    COALESCE(regexp_replace(was_price_display, '[^\d.]', '', 'g')) AS was_price,
    regexp_replace(price_display, '[^\d.]', '', 'g')::FLOAT AS current_price,
    selling_size,
    price_comparsion_display,
    unit_raw,
    unit_qty,
    CASE WHEN unit_qty IS NULL    THEN 1.0
    WHEN unit_raw = 'g'      THEN unit_qty / 1000
    WHEN unit_raw = 'kg'     THEN unit_qty
    WHEN unit_raw = 'ml'     THEN unit_qty / 1000
    WHEN unit_raw = 'l'      THEN unit_qty
    WHEN unit_raw = 'cl'     THEN unit_qty / 100
    ELSE unit_qty
    END AS unit_qty_normalised,
    CASE unit_raw
            WHEN 'g'  THEN 'kg'
            WHEN 'kg' THEN 'kg'
            WHEN 'ml' THEN 'l'
            WHEN 'l'  THEN 'l'
            WHEN 'litres' THEN 'l'
            WHEN 'litre' THEN 'l'
            WHEN 'cl' THEN 'l'
            ELSE 'each'
        END AS unit_normalised,
    CASE
    WHEN selling_size IS NOT NULL THEN 'selling_size'
    WHEN price_comparsion_display IS NOT NULL THEN 'price_comparison display'
    WHEN REGEXP_EXTRACT(title, '\d+\.?\d*\s*[a-zA-Z]+') IS NOT NULL THEN 'title'
    ELSE 'each' END AS size_source
FROM stg)

SELECT *,  ROUND(current_price / NULLIF(unit_qty_normalised, 0),2) AS price_per_unit_normalised
FROM extracted
""").pl()  # .pl() converts to polars DataFrame


In [ ]:
df_int = conn.execute(r"""SELECT * FROM int_aldi
""").pl()  # .pl() converts to polars DataFrame


In [ ]:
df.filter(pl.col("unit_normalised")=="each").head(20)

In [ ]:
list(df.select(pl.col('unit_raw').unique())["unit_raw"])

In [ ]:
list(df.select(pl.col('unit_normalised').unique())["unit_normalised"])

In [ ]:
list(df.filter(pl.col('brand').str.contains(r"[^\w\s]")).select(pl.col("brand").unique())['brand'])

In [ ]:
df.filter((pl.col("quantity").is_null()) & (pl.col("unit_price").is_not_null()))